# 01 Fundamentals of Prompt Engineering

**Real-World Scenario**: Building an Enterprise Financial Earnings Analyst Assistant.

This notebook demonstrates Prompt Anatomy, Role Isolation (`System` vs `User`), Token Budget Calculations using `tiktoken`, and Multi-Provider LLM calling (`ChatOpenAI`, `OllamaLLM`) via LangChain.

## Step 1: Environment & API Key Configuration

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from root .env file
load_dotenv()

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))


## Step 2: Token Budget & API Cost Estimation (`tiktoken`)

In [ ]:
import tiktoken

def calculate_token_budget(system_text: str, user_text: str, model_name: str = "gpt-4o-mini"):
    enc = tiktoken.encoding_for_model(model_name)
    sys_tokens = len(enc.encode(system_text))
    user_tokens = len(enc.encode(user_text))
    total_tokens = sys_tokens + user_tokens
    
    # Pricing: $2.50 per 1,000,000 input tokens
    cost = (total_tokens / 1_000_000) * 2.50
    
    print(f"=== Token Budget Analysis ({model_name}) ===")
    print(f"System Prompt Tokens: {sys_tokens}")
    print(f"User Query Tokens:   {user_tokens}")
    print(f"Total Input Tokens:  {total_tokens}")
    print(f"Estimated Input Cost: ${cost:.6f}")
    return total_tokens

sys_prompt = "You are a Senior Financial Analyst. Extract Q3 revenue and growth metrics strictly from context. Do NOT include preambles."
user_prompt = "Nvidia Q3 revenue reached $18.12 billion, up 206% YoY driven by Hopper GPU demand."
calculate_token_budget(sys_prompt, user_prompt)


## Step 3: Enterprise System Prompt Construction with LangChain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "{system_instruction}"),
    ("user", "Document Context:
{context}

Target Metric: {metric}")
])

formatted_prompt = prompt_template.format(
    system_instruction="You are an enterprise financial entity extraction engine. Output raw answer ONLY.",
    context="Nvidia Q3 revenue reached $18.12 billion, up 206% YoY.",
    metric="Revenue YoY Growth Rate"
)

print("--- Formatted LangChain System Prompt Sequence ---")
print(formatted_prompt)


## Step 4: Multi-Provider Model Execution (OpenAI & Ollama)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.llms import Ollama

print("=== 1. Executing via OpenAI ChatOpenAI ===")
try:
    if os.getenv("OPENAI_API_KEY"):
        llm_openai = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
        res_openai = llm_openai.invoke(formatted_prompt)
        print("ChatOpenAI Response:
", res_openai.content)
    else:
        print("OPENAI_API_KEY not found in .env.")
except Exception as e:
    print("OpenAI execution error:", e)

print("
=== 2. Executing via Local Ollama ===")
try:
    llm_ollama = Ollama(model="llama3", timeout=3)
    res_ollama = llm_ollama.invoke(formatted_prompt)
    print("Ollama Llama-3 Response:
", res_ollama)
except Exception as e:
    print("Ollama local execution skipped (Start server with 'ollama run llama3'):", e)
